In [139]:
import pandas as pd
import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots

База: загрузка датасета

In [140]:
data = pd.read_csv('data_ab.csv')

In [141]:
data.head()

,user_id,timestamp,group,landing_page,converted
0,851104,2025-01-21 22:11:48.556739,control,old_page,0
1,804228,2025-01-12 08:01:45.159739,control,old_page,0
2,661590,2025-01-11 16:55:06.154213,treatment,new_page,0
3,853541,2025-01-08 18:28:03.143765,treatment,new_page,0
4,864975,2025-01-21 01:52:26.210827,control,old_page,1


* `user_id` — уникальный идентификатор пользователя;

* `timestamp` — время посещения пользователем страницы;

* `group` — группа эксперимента, к которой был случайно отнесён пользователь:

  *   `control` — контрольная группа,
  *   `treatment` — экспериментальная группа;

* `landing_page` — версия лендинга, которую увидел пользователь:

  *   `old_page` — старая версия страницы,
  *   `new_page` — новая версия страницы;

* `converted` — бинарный признак, показывающий, совершил ли пользователь целевое действие (целевым действием является регистрация на курс).

Наша целевая переменная это совершение целевого действия пользователем.

In [142]:
data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 294478 entries, 0 to 294477
Data columns (total 5 columns):
 #   Column        Non-Null Count   Dtype 
---  ------        --------------   ----- 
 0   user_id       294478 non-null  int64 
 1   timestamp     294478 non-null  object
 2   group         294478 non-null  object
 3   landing_page  294478 non-null  object
 4   converted     294478 non-null  int64 
dtypes: int64(2), object(3)
memory usage: 11.2+ MB


In [143]:
data.isna().sum()

user_id         0
timestamp       0
group           0
landing_page    0
converted       0
dtype: int64

пропусков нет

In [144]:
mask = (data['group'] == 'treatment') & (data['landing_page'] == 'old_page') | (data['group'] == 'control') & (data['landing_page'] == 'new_page')

In [145]:
data[mask]

,user_id,timestamp,group,landing_page,converted
22,767017,2025-01-12 22:58:14.991443,control,new_page,0
240,733976,2025-01-11 15:11:16.407599,control,new_page,0
308,857184,2025-01-20 07:34:59.832626,treatment,old_page,0
327,686623,2025-01-09 14:26:40.734775,treatment,old_page,0
357,856078,2025-01-12 12:29:30.354835,treatment,old_page,0
...,...,...,...,...,...
294014,813406,2025-01-09 06:25:33.223301,treatment,old_page,0
294200,928506,2025-01-13 21:32:10.491309,control,new_page,0
294252,892498,2025-01-22 01:11:10.463211,treatment,old_page,0
294253,886135,2025-01-06 12:49:20.509403,control,new_page,0


3893 значений, которые противоречат нашему тесту(По замыслу A/B-теста контрольной группе (`control`) должна была показываться старая версия лендинга — `old_page`; экспериментальной группе (`treatment`) должна была показываться новая версия лендинга — `new_page`). Эти данные нужно удалить, так как они будут создавать шум и искажать значения

In [146]:
data = data[-mask]

In [147]:
data['timestamp'] = pd.to_datetime(data['timestamp'])
data['date'] = data['timestamp'].dt.date
data['weekday'] = data['timestamp'].dt.weekday

In [148]:
data.head()

,user_id,timestamp,group,landing_page,converted,date,weekday
0,851104,2025-01-21 22:11:48.556739,control,old_page,0,2025-01-21,1
1,804228,2025-01-12 08:01:45.159739,control,old_page,0,2025-01-12,6
2,661590,2025-01-11 16:55:06.154213,treatment,new_page,0,2025-01-11,5
3,853541,2025-01-08 18:28:03.143765,treatment,new_page,0,2025-01-08,2
4,864975,2025-01-21 01:52:26.210827,control,old_page,1,2025-01-21,1


Посмотрим разделение по классам

In [149]:
print(data['group'].value_counts()) #без принта почему-то криво выводит

group
treatment    145311
control      145274
Name: count, dtype: int64


In [150]:
print(data['converted'].value_counts())

converted
0    255832
1     34753
Name: count, dtype: int64


Целевое действие совершает меньше чем 20% пользователей.

In [151]:
size = len(data)

In [152]:
print(data['group'].value_counts()/size)

group
treatment    0.500064
control      0.499936
Name: count, dtype: float64


распределение групп почти 50/50

In [153]:
print(data['converted'].value_counts()/size)

converted
0    0.880403
1    0.119597
Name: count, dtype: float64


88% пользователей не совершают целевое действие независимо от группы

In [154]:
group_a = data[data['group'] == 'control']
group_b = data[data['group'] == 'treatment']

size_a = len(group_a)
size_b = len(group_b)

In [155]:
print(group_a['converted'].value_counts()/size_a)
print(group_b['converted'].value_counts()/size_b)

print(group_a['converted'].mean())
print(group_b['converted'].mean())

converted
0    0.879614
1    0.120386
Name: count, dtype: float64
converted
0    0.881193
1    0.118807
Name: count, dtype: float64
0.1203863045004612
0.11880724790277405


Исходя из полученных данных мы можем предположить, что новая версия страницы сделала только хуже. Она уменьшает конверсию на 1%.

Но делать такие поспешные выводы мы не можем, так как у нас нет статистически значимых оснований так говорить.

Посмотрим на колонку с временем.

In [156]:
group_by_date = data.groupby(['date', 'group'])['user_id'].count()

In [157]:
group_by_date/size

date        group    
2025-01-02  control      0.009839
            treatment    0.009818
2025-01-03  control      0.022678
            treatment    0.022775
2025-01-04  control      0.022637
            treatment    0.022510
2025-01-05  control      0.022117
            treatment    0.022386
2025-01-06  control      0.022733
            treatment    0.023219
2025-01-07  control      0.022727
            treatment    0.022744
2025-01-08  control      0.023012
            treatment    0.023057
2025-01-09  control      0.022809
            treatment    0.022764
2025-01-10  control      0.022899
            treatment    0.023043
2025-01-11  control      0.023016
            treatment    0.022964
2025-01-12  control      0.022444
            treatment    0.022840
2025-01-13  control      0.022548
            treatment    0.022396
2025-01-14  control      0.022534
            treatment    0.022713
2025-01-15  control      0.023105
            treatment    0.022537
2025-01-16  control      0

Группы однородны также по дням

Каждый день по 4

In [158]:
group_by_date = data.groupby('date')['converted'].mean()

In [159]:
group_by_date

date
2025-01-02    0.122724
2025-01-03    0.113795
2025-01-04    0.119293
2025-01-05    0.119084
2025-01-06    0.119449
2025-01-07    0.118595
2025-01-08    0.119818
2025-01-09    0.118855
2025-01-10    0.119625
2025-01-11    0.116982
2025-01-12    0.122198
2025-01-13    0.114089
2025-01-14    0.122984
2025-01-15    0.117017
2025-01-16    0.120509
2025-01-17    0.125048
2025-01-18    0.124799
2025-01-19    0.118583
2025-01-20    0.116476
2025-01-21    0.120896
2025-01-22    0.118583
2025-01-23    0.123380
2025-01-24    0.119839
Name: converted, dtype: float64

Мы получили общую конверсию по датам независимо от группы

In [160]:
conversion = go.Scatter(x=group_by_date.index, y=group_by_date.values)

fig = go.Figure(conversion, layout_title_text="График конверсии по датам, независимо от группы")

fig.show()

In [161]:
group_A_by_date = group_a.groupby('date')['converted'].mean()

In [162]:
group_A_by_date

date
2025-01-02    0.125568
2025-01-03    0.113809
2025-01-04    0.121922
2025-01-05    0.123230
2025-01-06    0.115350
2025-01-07    0.120987
2025-01-08    0.118887
2025-01-09    0.119644
2025-01-10    0.112864
2025-01-11    0.118870
2025-01-12    0.122048
2025-01-13    0.116911
2025-01-14    0.126756
2025-01-15    0.120494
2025-01-16    0.121833
2025-01-17    0.122865
2025-01-18    0.124807
2025-01-19    0.119945
2025-01-20    0.115243
2025-01-21    0.125945
2025-01-22    0.119163
2025-01-23    0.125670
2025-01-24    0.118007
Name: converted, dtype: float64

In [163]:
group_B_by_date = group_b.groupby('date')['converted'].mean()

In [164]:
group_B_by_date

date
2025-01-02    0.119874
2025-01-03    0.113781
2025-01-04    0.116649
2025-01-05    0.114988
2025-01-06    0.123462
2025-01-07    0.116205
2025-01-08    0.120746
2025-01-09    0.118065
2025-01-10    0.126344
2025-01-11    0.115091
2025-01-12    0.122344
2025-01-13    0.111248
2025-01-14    0.119242
2025-01-15    0.113452
2025-01-16    0.119175
2025-01-17    0.127256
2025-01-18    0.124792
2025-01-19    0.117216
2025-01-20    0.117682
2025-01-21    0.115701
2025-01-22    0.118009
2025-01-23    0.121061
2025-01-24    0.121706
Name: converted, dtype: float64

In [165]:
figA = go.Scatter(x=group_A_by_date.index, y=group_A_by_date.values, name='Группа A')
figB = go.Scatter(x=group_B_by_date.index, y=group_B_by_date.values, name='Группа B')

data_new = [figA, figB]

fig = go.Figure(data_new, layout_title_text="Графики конверсии по датам для групп A и B")

fig.show()

In [166]:
group_by_weekday = data.groupby('weekday')['converted'].mean()

In [167]:
days = {0: 'Понедельник', 1: 'Вторник', 2: 'Среда', 3: 'Четверг', 4: 'Пятница', 5: 'Суббота', 6: 'Воскресенье'}

In [168]:
group_by_weekday

weekday
0    0.116691
1    0.120822
2    0.118477
3    0.121149
4    0.119538
5    0.120334
6    0.119961
Name: converted, dtype: float64

In [169]:
group_by_weekday.index = group_by_weekday.index.map(days)

In [170]:
conversion = go.Scatter(x=group_by_weekday.index, y=group_by_weekday.values)

fig = go.Figure(conversion, layout_title_text="График конверсии по дням недели, независимо от группы")

fig.show()

In [171]:
group_A_by_weekday = group_a.groupby('weekday')['converted'].mean()

In [172]:
group_A_by_weekday

weekday
0    0.115834
1    0.124567
2    0.119518
3    0.122795
4    0.116748
5    0.121835
6    0.121729
Name: converted, dtype: float64

In [173]:
group_A_by_weekday.index = group_A_by_weekday.index.map(days)

In [174]:
group_B_by_weekday = group_b.groupby('weekday')['converted'].mean()

In [175]:
group_B_by_weekday

weekday
0    0.117538
1    0.117052
2    0.117431
3    0.119491
4    0.122339
5    0.118837
6    0.118209
Name: converted, dtype: float64

In [176]:
group_B_by_weekday.index = group_B_by_weekday.index.map(days)

In [177]:
figA = go.Scatter(x=group_A_by_weekday.index, y=group_A_by_weekday.values, name='Группа A')
figB = go.Scatter(x=group_B_by_weekday.index, y=group_B_by_weekday.values, name='Группа B')

data_new = [figA, figB]

fig = go.Figure(data_new, layout_title_text="Графики конверсии по дням недели для групп A и B")

fig.show()